In [ ]:
import sys
sys.path.append("..")   # add main_folder to path


import geopandas as gpd
import itertools
import pyarrow.dataset as pds
from tqdm import tqdm
import xarray as xr
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Optional
from tqdm_joblib import tqdm_joblib
import more_itertools

from joblib import Parallel, delayed
import pyarrow.compute as pc
from shapely.prepared import prep


import tomllib
from loguru import logger

from src.climate_data_merging.climate_data_merging import process_month
from src.climate_data_merging.climate_data_merging_utils import get_catherina_coefs


In [ ]:
def plot_variable_distributions_by_period(
    df: pd.DataFrame,
    date_col: str = "date",
    variable_cols: Optional[List[str]] = None,  # only for wide format
    long_format: bool = False,                  # set True if df already has 'variable' and 'value'
    value_col: str = "value",
    variable_name_col: str = "variable",
    dropna: bool = True,
    nbins: int = 40,
    alpha: float = 0.5,
    figsize=(6, 4),
    save_dir: Optional[str] = None,             # e.g. "figs"; if None, don't save
):
    """
    Compare distributions of 4 climate variables across three time periods:
      - 2025–2049 inclusive
      - 2050–2074 inclusive
      - >= 2075
    Assumes multiple locations; all locations are pooled within each period.
    Creates one figure per variable with overlaid histograms for the 3 periods.
    """

    # 1) Parse dates and build period labels
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])

    # Define period edges
    p1_start = pd.Timestamp("1990-01-01")
    p1_end   = pd.Timestamp("2049-12-31")
    p2_start = pd.Timestamp("2050-01-01")
    p2_end   = pd.Timestamp("2074-12-31")
    p3_start = pd.Timestamp("2075-01-01")

    def _label_period(ts: pd.Timestamp) -> str:
        if pd.isna(ts):
            return "Unknown"
        if p1_start <= ts <= p1_end:
            return "2025–2049"
        if p2_start <= ts <= p2_end:
            return "2050–2074"
        if ts >= p3_start:
            return "≥ 2075"
        return "< 2025"  # falls before analysis window; we’ll drop later

    out["period"] = out[date_col].map(_label_period)

    # Keep only the three requested periods
    out = out[out["period"].isin(["2025–2049", "2050–2074", "≥ 2075"])]

    # 2) Reshape to long if needed
    if not long_format:
        if variable_cols is None:
            # Try to infer: pick non-date numeric columns (you can pass explicit list instead)
            variable_cols = [c for c in out.columns
                             if c not in {date_col, "period"} and np.issubdtype(out[c].dtype, np.number)]
        long = out.melt(
            id_vars=[date_col, "period"],
            value_vars=variable_cols,
            var_name=variable_name_col,
            value_name=value_col
        )
    else:
        # Expect columns: date_col, variable_name_col, value_col (+ others allowed)
        long = out[[date_col, "period", variable_name_col, value_col]].copy()

    # 3) Clean values
    if dropna:
        long = long.dropna(subset=[value_col])

    # 4) Plot – one figure per variable, overlay the three periods
    variables = long[variable_name_col].unique()

    for var in variables:
        sub = long[long[variable_name_col] == var]

        # Prepare data arrays per period
        data_p1 = sub.loc[sub["period"] == "2025–2049", value_col].to_numpy()
        data_p2 = sub.loc[sub["period"] == "2050–2074", value_col].to_numpy()
        data_p3 = sub.loc[sub["period"] == "≥ 2075",     value_col].to_numpy()

        # Common bins across periods for fair comparison
        # Use combined min/max with small padding
        all_vals = np.concatenate([x for x in [data_p1, data_p2, data_p3] if x.size > 0]) \
                   if (data_p1.size + data_p2.size + data_p3.size) > 0 else np.array([])
        if all_vals.size == 0:
            print(f"[warn] No data to plot for variable '{var}'. Skipping.")
            continue
        vmin, vmax = np.nanmin(all_vals), np.nanmax(all_vals)
        if vmin == vmax:
            # Avoid degenerate bins
            vmin, vmax = vmin - 0.5, vmax + 0.5
        bins = np.linspace(vmin, vmax, nbins)

        # Create figure
        plt.figure(figsize=figsize)
        # Overlaid histograms (density normalized)
        if data_p1.size:
            plt.hist(data_p1, bins=bins, density=True, alpha=alpha, label="2025–2049")
        if data_p2.size:
            plt.hist(data_p2, bins=bins, density=True, alpha=alpha, label="2050–2074")
        if data_p3.size:
            plt.hist(data_p3, bins=bins, density=True, alpha=alpha, label="≥ 2075")

        plt.title(f"Distribution of {var} by period")
        plt.xlabel(var)
        plt.ylabel("Density")
        plt.legend()
        plt.tight_layout()

        if save_dir is not None:
            import os
            os.makedirs(save_dir, exist_ok=True)
            fname = os.path.join(save_dir, f"{var}_distribution_by_period.png")
            plt.savefig(fname, dpi=150)
        # Show each figure separately if running interactively
        plt.show()

    print("Done. Created one histogram figure per variable.")

In [ ]:
config_path = "../config.toml"
with open(
    config_path,
    "rb",
) as f:  # Open the file in binary mode
    config_files = tomllib.load(f)

In [ ]:
main_config = config_files["main_params"]
gen_config = config_files["generation"]
intens_config = config_files["intensification"]
seeds = list(range(gen_config["n_seeds"]))

catherina_fit_path = ".." / Path(gen_config["fit_dir"]) / "Catherina_fit.db"
coefs = get_catherina_coefs(catherina_fit_path=catherina_fit_path)
cyclones_dir = Path(gen_config["synthetic_tracks_dir"])

data_dir = ".." / Path(main_config["data_dir"])


# Climate variables
cmip_dir = ".." / Path(main_config["data_dir"]) / "cmip6_data_with_hurs/"

ne_10m_coastline_zip = ".." / Path(main_config["data_dir"]) / "ne_10m_coastline.zip"
ne_10m_land_zip = ".." /Path(main_config["data_dir"]) / "ne_10m_land.zip"
model_exp_pbar = tqdm(
    list(itertools.product(main_config["models"], main_config["experiments"]))
)
land = gpd.read_file(ne_10m_land_zip).union_all()

In [ ]:
model="ACCESS-CM2"
experiment="ssp585"

clim_ds = xr.open_zarr(cmip_dir / model / experiment)
model_exp_pbar.set_postfix(model=model, experiment=experiment)

In [ ]:
tracks_dir = '..' / Path(main_config["data_dir"]) / "catherina_ssp585" / "tracks"
tracks = pds.dataset(tracks_dir, format="parquet", partitioning="hive")

# scanner = pds.Scanner.from_dataset(
#         tracks,
#         filter=(pc.is_valid(pc.field("lat"))
#             & pc.is_valid(pc.field("lon"))
#         ),
#     ).to_table()

# tracks = pds.dataset(scanner)

In [ ]:
tracks_with_env_path = (
         data_dir/ f"catherina_ssp585/tracks_with_env4/{model}/{experiment}/"
    )

yearmonth_batches = list(itertools.product(range(2025, 2100), range(1, 12+1))) # or 4/8 depending on RAM
tasks = []
for yearmonth in yearmonth_batches:
    tasks.append(yearmonth)


Parallel(n_jobs=8, prefer="threads")(
            delayed(process_month)(
                year=year,
                month=month,
                clim_ds=clim_ds,
                model=model,
                experiment=experiment,
                tracks=tracks,
                save_dir=tracks_with_env_path,
            )
            for year, month in tqdm(tasks, desc="year-month")
        )

In [ ]:
tracks_with_env_path = (
         data_dir/ f"catherina_ssp585/tracks_with_env4/{model}/{experiment}/"
    )
tracks_with_climate_data = pds.dataset(
            tracks_with_env_path,
            format="parquet",
            partitioning="hive",
        )

In [ ]:
plot_variable_distributions_by_period(
    tracks_with_climate_data.to_table().to_pandas(),
    date_col="datetime",
    variable_cols=["tos","ta", "psl", "hur"],  # your 4 variables (wide format)
    long_format=False,          # set True if your df already has 'variable' and 'value'
    nbins=40,
    save_dir=None               # or "figs"
)
